In [ ]:
#import dataset from roboflow

In [ ]:
import cv2
import numpy as np
import yaml
import time
import json
from pathlib import Path
from ultralytics import YOLO

# 1. HÀM CHUẨN HÓA TÊN CLASS (Để model 1, model 2 và dataset gọi chung 1 tên)
def canonicalize_class_name(name: str) -> str:
    normalized = name.strip().lower().replace("-", "_").replace(" ", "_")
    alias_map = {
        "head_light": "head_light", "headlight": "head_light", "front_light": "head_light", 
        "car_front": "head_light", "front_car": "head_light",
        "rear_light": "rear_light", "rearlight": "rear_light", "back_light": "rear_light", 
        "car_rear": "rear_light", "rear_car": "rear_light"
    }
    return alias_map.get(normalized, normalized)

# 2. CLASS ENSEMBLE TRỘN MODEL
class EnsembleDetector:
    def __init__(self, primary_model_path, secondary_model_path):
        self.primary_model = YOLO(primary_model_path)
        self.secondary_model = YOLO(secondary_model_path)

    def fuse_detections(
        self,
        dets1,
        dets2,
        iou_thresh=0.5,
        method="nms",
        score_mode="max"
    ):
        """
        method:
            - "nms": NMS thường, giữ box confidence cao nhất.
            - "weighted_nms": gộp box overlap cùng class bằng trung bình có trọng số confidence.

        score_mode cho weighted_nms:
            - "max": confidence sau fuse = max confidence trong cụm box.
            - "mean": confidence sau fuse = mean confidence trong cụm box.
            - "weighted_mean": confidence sau fuse = weighted mean confidence.
        """

        all_dets = dets1 + dets2

        if not all_dets:
            return []

        if method not in ["nms", "weighted_nms"]:
            raise ValueError("method must be 'nms' or 'weighted_nms'")

        fused = []

        class_names = sorted(set(d["cls_name"] for d in all_dets))

        for cls_name in class_names:
            cls_dets = [d for d in all_dets if d["cls_name"] == cls_name]
            cls_dets = sorted(cls_dets, key=lambda x: x["conf"], reverse=True)

            while len(cls_dets) > 0:
                base_det = cls_dets[0]

                matched = []
                remaining = []

                for det in cls_dets:
                    iou = compute_iou(base_det["box"], det["box"])

                    if iou >= iou_thresh:
                        matched.append(det)
                    else:
                        remaining.append(det)

                if method == "nms":
                    fused.append(base_det)

                elif method == "weighted_nms":
                    boxes = np.array([d["box"] for d in matched], dtype=np.float64)
                    confs = np.array([d["conf"] for d in matched], dtype=np.float64)
                
                    # Diagnostic
                    base_det["cluster_size"] = len(matched)
                
                    weights = confs / (confs.sum() + 1e-12)
                    fused_box = np.sum(boxes * weights[:, None], axis=0)
                
                    if score_mode == "max":
                        fused_conf = float(np.max(confs))
                    elif score_mode == "mean":
                        fused_conf = float(np.mean(confs))
                    elif score_mode == "weighted_mean":
                        fused_conf = float(np.sum(confs * weights))
                    else:
                        raise ValueError("score_mode must be 'max', 'mean', or 'weighted_mean'")
                
                    fused.append({
                        "box": fused_box,
                        "cls": base_det["cls"],
                        "cls_name": cls_name,
                        "conf": fused_conf,
                        "cluster_size": len(matched)
                    })

                cls_dets = remaining

        fused = sorted(fused, key=lambda x: x["conf"], reverse=True)
        return fused

# 3. HÀM ĐỌC NHÃN YOLO (.txt)
def parse_yolo_label(label_path: Path, img_w: int, img_h: int, class_names_dict: dict) -> list:
    gt_boxes = []
    if not label_path.exists(): return gt_boxes
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
        
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_id = int(float(parts[0]))
            cx, cy, w, h = map(float, parts[1:5])
            
            # Giải mã tọa độ chuẩn hóa (0-1) về Pixel thật
            abs_cx, abs_cy = cx * img_w, cy * img_h
            abs_w, abs_h = w * img_w, h * img_h
            
            x1 = abs_cx - abs_w / 2
            y1 = abs_cy - abs_h / 2
            x2 = abs_cx + abs_w / 2
            y2 = abs_cy + abs_h / 2
            
            cls_name = canonicalize_class_name(class_names_dict.get(cls_id, str(cls_id)))
            gt_boxes.append({
                "bbox": np.array([x1, y1, x2, y2], dtype=np.float64),
                "cls_name": cls_name
            })
    return gt_boxes

# 4. CÁC HÀM TÍNH TOÁN METRICS CƠ BẢN
def compute_iou(box_a, box_b):
    x1 = max(float(box_a[0]), float(box_b[0]))
    y1 = max(float(box_a[1]), float(box_b[1]))
    x2 = min(float(box_a[2]), float(box_b[2]))
    y2 = min(float(box_a[3]), float(box_b[3]))
    inter_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union_area = (max(0.0, float(box_a[2] - box_a[0])) * max(0.0, float(box_a[3] - box_a[1])) + 
                  max(0.0, float(box_b[2] - box_b[0])) * max(0.0, float(box_b[3] - box_b[1])) - inter_area)
    return inter_area / union_area if union_area > 0 else 0.0

def compute_ap(recalls, precisions):
    if not recalls or not precisions: return 0.0
    recalls = np.array([0.0] + recalls + [1.0])
    precisions = np.array([0.0] + precisions + [0.0])
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    indices = np.where(recalls[1:] != recalls[:-1])[0]
    return float(np.sum((recalls[indices + 1] - recalls[indices]) * precisions[indices + 1]))

def get_model_class_name(model, cls_id):
    names = model.names

    if isinstance(names, dict):
        raw_name = names.get(int(cls_id), str(cls_id))
    else:
        raw_name = names[int(cls_id)]

    return canonicalize_class_name(raw_name)


def run_single_model(model, image, conf_threshold, imgsz=1536):
    dets = []

    results = model.predict(
        image,
        conf=conf_threshold,
        imgsz=imgsz,
        verbose=False
    )

    if not results or results[0].boxes is None:
        return dets

    boxes = results[0].boxes
    xyxy = boxes.xyxy.cpu().numpy() if boxes.xyxy is not None else []
    classes = boxes.cls.int().cpu().tolist() if boxes.cls is not None else []
    confs = boxes.conf.cpu().tolist() if boxes.conf is not None else []

    for b, c_id, conf in zip(xyxy, classes, confs):
        cls_name = get_model_class_name(model, c_id)

        dets.append({
            "box": b.astype(np.float64),
            "cls": int(c_id),
            "cls_name": cls_name,
            "conf": float(conf)
        })

    return dets

In [ ]:
import pandas as pd
from IPython.display import display

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN KAGGLE + ROBOFLOW (YOLO)
# ==========================================
# Thay tên thư mục ('valid' hoặc 'test') tùy vào thư mục bạn muốn đánh giá
SPLIT_NAME = "train" 


CONFIG = {
    "primary_model": "YOUR_MODEL_PATH.pt",  # Thay YOUR_MODEL_PATH bằng đường dẫn thực tế đến model của bạn
    "secondary_model": "YOUR_MODEL_PATH.pt",  # Thay YOUR_MODEL_PATH bằng đường dẫn thực tế đến model của bạn

    "yaml_path": f"{dataset.location}/data.yaml",

    # Nên dùng SPLIT_NAME thay vì hard-code train
    "image_dir": f"{dataset.location}/{SPLIT_NAME}/images",
    "label_dir": f"{dataset.location}/{SPLIT_NAME}/labels",

    "imgsz": 1536,

    # Sweep confidence
    "conf_sweep": [0.10, 0.15, 0.20, 0.25, 0.30],

    # IoU dùng cho NMS / Weighted NMS
    "fusion_iou_sweep": [0.35, 0.40, 0.45, 0.50, 0.55, 0.60],

    # IoU dùng để đánh giá mAP@50, Precision, Recall
    "eval_iou_threshold": 0.50,

    "optimize": "map50",
    "output_json": "/kaggle/working/ensemble_metrics.json",
    "output_csv": "/kaggle/working/ensemble_per_class.csv"
}

# ==========================================
# 2. ĐỌC DATA.YAML ĐỂ LẤY THÔNG TIN CLASS
# ==========================================
with open(CONFIG["yaml_path"], 'r') as f:
    data_yaml = yaml.safe_load(f)

# Dataset Roboflow trả về tên class dạng list hoặc dict, ta đưa về dict {0: 'class_A'}
if isinstance(data_yaml['names'], list):
    dataset_classes = {i: name for i, name in enumerate(data_yaml['names'])}
else:
    dataset_classes = data_yaml['names']

valid_class_names = sorted(list(set([canonicalize_class_name(n) for n in dataset_classes.values()])))
print(f"📌 Các class cần đánh giá: {valid_class_names}")

# ==========================================
# 3. QUÁ TRÌNH CHẠY ENSEMBLE & ĐÁNH GIÁ
# ==========================================
methods_to_compare = ["nms", "weighted_nms"]

all_results = {}

detector = EnsembleDetector(CONFIG["primary_model"], CONFIG["secondary_model"])

image_paths = (
    list(Path(CONFIG["image_dir"]).glob("*.jpg")) +
    list(Path(CONFIG["image_dir"]).glob("*.jpeg")) +
    list(Path(CONFIG["image_dir"]).glob("*.png"))
)

print(f"🚀 Bắt đầu evaluate trên {len(image_paths)} ảnh...")

for fuse_method in methods_to_compare:
    print("\n" + "=" * 60)
    print(f"🔍 Đang evaluate method: {fuse_method}")
    print("=" * 60)

    best_summary = None
    best_rows = []
    best_conf = None
    best_fusion_iou = None

    for current_conf in CONFIG["conf_sweep"]:
        for current_fusion_iou in CONFIG["fusion_iou_sweep"]:

            print(
                f"  👉 Đang test method={fuse_method}, "
                f"conf={current_conf}, fusion_iou={current_fusion_iou}"
            )

            stats_by_class = {
                c: {
                    "num_gt": 0,
                    "preds": []
                }
                for c in valid_class_names
            }

            for img_path in image_paths:
                image = cv2.imread(str(img_path))

                if image is None:
                    continue

                h, w = image.shape[:2]

                label_path = Path(CONFIG["label_dir"]) / f"{img_path.stem}.txt"
                gt_boxes = parse_yolo_label(label_path, w, h, dataset_classes)

                gt_by_class = {c: [] for c in valid_class_names}

                for gt in gt_boxes:
                    cls_name = gt["cls_name"]

                    if cls_name in gt_by_class:
                        gt_by_class[cls_name].append(gt["bbox"])
                        stats_by_class[cls_name]["num_gt"] += 1

                dets1 = run_single_model(
                    detector.primary_model,
                    image,
                    current_conf,
                    imgsz=CONFIG["imgsz"]
                )

                dets2 = run_single_model(
                    detector.secondary_model,
                    image,
                    current_conf,
                    imgsz=CONFIG["imgsz"]
                )

                fused_dets = detector.fuse_detections(
                    dets1,
                    dets2,
                    iou_thresh=current_fusion_iou,
                    method=fuse_method,
                    score_mode="max"
                )

                preds_by_class = {c: [] for c in valid_class_names}

                for det in fused_dets:
                    cls_name = det["cls_name"]

                    if cls_name in preds_by_class:
                        preds_by_class[cls_name].append(det)

                for cls_name in valid_class_names:
                    gts = gt_by_class[cls_name]
                    preds = sorted(
                        preds_by_class[cls_name],
                        key=lambda x: x["conf"],
                        reverse=True
                    )

                    gt_used = [False] * len(gts)

                    for pred in preds:
                        best_iou = 0.0
                        best_idx = -1

                        for idx, gt_box in enumerate(gts):
                            if gt_used[idx]:
                                continue

                            iou = compute_iou(pred["box"], gt_box)

                            if iou > best_iou:
                                best_iou = iou
                                best_idx = idx

                        is_tp = 1 if (
                            best_idx >= 0 and
                            best_iou >= CONFIG["eval_iou_threshold"]
                        ) else 0

                        if is_tp:
                            gt_used[best_idx] = True

                        stats_by_class[cls_name]["preds"].append({
                            "conf": pred["conf"],
                            "tp": is_tp,
                            "fp": 1 - is_tp
                        })

            rows = []
            aps = []
            precisions = []
            recalls = []

            total_tp = 0
            total_fp = 0
            total_gt = 0

            for cls_name in valid_class_names:
                preds = sorted(
                    stats_by_class[cls_name]["preds"],
                    key=lambda x: x["conf"],
                    reverse=True
                )

                num_gt = stats_by_class[cls_name]["num_gt"]

                cum_tp = 0
                cum_fp = 0

                precision_curve = []
                recall_curve = []

                for pred in preds:
                    cum_tp += pred["tp"]
                    cum_fp += pred["fp"]

                    precision_i = (
                        cum_tp / (cum_tp + cum_fp)
                        if (cum_tp + cum_fp)
                        else 0.0
                    )

                    recall_i = (
                        cum_tp / num_gt
                        if num_gt
                        else 0.0
                    )

                    precision_curve.append(precision_i)
                    recall_curve.append(recall_i)

                ap50 = compute_ap(recall_curve, precision_curve)

                final_tp = sum(p["tp"] for p in preds)
                final_fp = sum(p["fp"] for p in preds)
                final_fn = num_gt - final_tp

                precision = (
                    final_tp / (final_tp + final_fp)
                    if (final_tp + final_fp)
                    else 0.0
                )

                recall = (
                    final_tp / num_gt
                    if num_gt
                    else 0.0
                )

                aps.append(ap50)
                precisions.append(precision)
                recalls.append(recall)

                total_tp += final_tp
                total_fp += final_fp
                total_gt += num_gt

                rows.append({
                    "Method": fuse_method,
                    "Conf": current_conf,
                    "Fusion IoU": current_fusion_iou,
                    "Eval IoU": CONFIG["eval_iou_threshold"],
                    "Class": cls_name,
                    "Targets": num_gt,
                    "TP": final_tp,
                    "FP": final_fp,
                    "FN": final_fn,
                    "Precision": round(precision, 4),
                    "Recall": round(recall, 4),
                    "mAP@50": round(ap50, 4)
                })

            map50 = float(np.mean(aps)) if aps else 0.0
            macro_precision = float(np.mean(precisions)) if precisions else 0.0
            macro_recall = float(np.mean(recalls)) if recalls else 0.0

            micro_precision = (
                total_tp / (total_tp + total_fp)
                if (total_tp + total_fp)
                else 0.0
            )

            micro_recall = (
                total_tp / total_gt
                if total_gt
                else 0.0
            )

            f1 = (
                2 * macro_precision * macro_recall /
                (macro_precision + macro_recall + 1e-16)
            )

            summary = {
                "Method": fuse_method,
                "Best Conf": current_conf,
                "Best Fusion IoU": current_fusion_iou,
                "Eval IoU": CONFIG["eval_iou_threshold"],
                "mAP@50": round(map50, 4),
                "Macro Precision": round(macro_precision, 4),
                "Macro Recall": round(macro_recall, 4),
                "Micro Precision": round(micro_precision, 4),
                "Micro Recall": round(micro_recall, 4),
                "F1": round(f1, 4),
                "TP": total_tp,
                "FP": total_fp,
                "GT": total_gt
            }

            if CONFIG["optimize"] == "map50":
                current_score = summary["mAP@50"]
                best_score = -1 if best_summary is None else best_summary["mAP@50"]

            elif CONFIG["optimize"] == "f1":
                current_score = summary["F1"]
                best_score = -1 if best_summary is None else best_summary["F1"]

            elif CONFIG["optimize"] == "recall":
                current_score = summary["Macro Recall"]
                best_score = -1 if best_summary is None else best_summary["Macro Recall"]

            else:
                raise ValueError("CONFIG['optimize'] must be 'map50', 'f1', or 'recall'")

            if best_summary is None or current_score > best_score:
                best_summary = summary
                best_rows = rows
                best_conf = current_conf
                best_fusion_iou = current_fusion_iou

    all_results[fuse_method] = {
        "summary": best_summary,
        "rows": best_rows,
        "best_conf": best_conf,
        "best_fusion_iou": best_fusion_iou
    }
# ==========================================
# 4. IN BÁO CÁO CUỐI CÙNG
# ==========================================
summary_df = pd.DataFrame([
    result["summary"]
    for result in all_results.values()
])

print("\n" + "=" * 60)
print("🏆 SO SÁNH NMS THƯỜNG VS WEIGHTED NMS")
print("=" * 60)

display(summary_df.sort_values("mAP@50", ascending=False))

for method_name, result in all_results.items():
    print("\n" + "=" * 60)
    print(f"📊 CHI TIẾT TỪNG CLASS - {method_name}")
    print("=" * 60)

    df_class = pd.DataFrame(result["rows"])
    display(df_class)

    output_path = f"/kaggle/working/ensemble_per_class_{method_name}.csv"
    df_class.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

output_json = "/kaggle/working/ensemble_nms_vs_weighted_nms.json"

json_output = {
    method_name: {
        "summary": result["summary"],
        "rows": result["rows"]
    }
    for method_name, result in all_results.items()
}

with open(output_json, "w") as f:
    json.dump(json_output, f, indent=2)

print(f"\nSaved JSON: {output_json}")